# Results Summary

In [1]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import wilcoxon

P = Path("processed")
stations = ["Bhaktapur", "Khumaltar", "Ratnapark", "Shankhapark"]
horizons = ["24", "48", "72"]

In [2]:
def paired_test(diff, n_boot=10000, seed=0):
    diff = np.asarray(diff)
    rng = np.random.default_rng(seed)
    boot = rng.choice(diff, (n_boot, len(diff))).mean(axis=1)
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {"n": len(diff), "mean": diff.mean().round(3), "ci_low": lo.round(3), "ci_high": hi.round(3),
            "wilcoxon_p": wilcoxon(diff).pvalue.round(4), "positive": int((diff > 0).sum())}

## 1. Regression: test MAE, 3 seeds

In [3]:
# test MAE for 3 models x 3 seeds (42, 43, 44)
rows = []
for s in stations:
    for seed in (42, 43, 44):
        f = f"reg_{s}_tuned300_eval.json" if seed == 42 else f"reg_{s}_tuned300_seed{seed}_eval.json"
        r = json.load(open(P / f))
        for h in horizons:
            rows.append({"model": "lstm", "seed": seed, "station": s, "h": h, "mae": r[h]["mae"], "rmse": r[h]["rmse"]})

for arm in ("graph", "nograph"):
    for seed in (42, 43, 44):
        r = json.load(open(P / f"ablate_reg_{arm}_seed{seed}.json"))["results"]
        for s in stations:
            for h in horizons:
                rows.append({"model": arm, "seed": seed, "station": s, "h": h, "mae": r[s][h]["mae"], "rmse": r[s][h]["rmse"]})

d = pd.DataFrame(rows)
print(d.groupby(["station", "h", "model"]).mae.mean().unstack().round(2))
print()
print(d.groupby("model")[["mae", "rmse"]].mean().round(2))

model           graph   lstm  nograph
station     h                        
Bhaktapur   24  16.30  14.37    14.80
            48  17.05  15.69    16.03
            72  17.39  16.45    16.45
Khumaltar   24  11.02  10.28    10.18
            48  12.12  11.41    11.23
            72  12.97  12.75    12.37
Ratnapark   24  15.00  15.40    13.91
            48  16.53  16.73    15.14
            72  18.39  17.94    16.78
Shankhapark 24  12.85  12.47    12.47
            48  13.80  13.44    13.34
            72  13.86  13.54    13.40

           mae   rmse
model                
graph    14.77  20.27
lstm     14.21  20.25
nograph  13.84  19.13


## 1b. Regression: test RMSE

In [4]:
print(d.groupby(["station", "h", "model"]).rmse.mean().unstack().round(2))

model           graph   lstm  nograph
station     h                        
Bhaktapur   24  23.21  21.38    21.10
            48  23.52  22.52    22.21
            72  23.36  22.84    22.32
Khumaltar   24  15.29  15.68    14.50
            48  16.48  17.07    15.56
            72  17.03  18.08    16.35
Ratnapark   24  21.15  23.25    19.96
            48  21.24  22.05    19.35
            72  22.56  22.92    20.57
Shankhapark 24  19.69  18.57    18.85
            48  20.23  19.51    19.66
            72  19.52  19.16    19.12


## 2. Regression ablation: no graph minus graph

Negative means no-graph is better.

In [5]:
# paired by seed: negative means no-graph is better
p = d.pivot_table(index=["station", "h", "seed"], columns="model", values="mae")
diff = (p["nograph"] - p["graph"]).groupby(["station", "h"]).agg(["mean", "std"]).round(2)
print(diff)
print("no-graph better in", (diff["mean"] < 0).sum(), "of", len(diff), "cells")

                mean   std
station     h             
Bhaktapur   24 -1.50  0.78
            48 -1.02  0.51
            72 -0.94  0.44
Khumaltar   24 -0.83  0.21
            48 -0.89  0.15
            72 -0.60  0.03
Ratnapark   24 -1.08  0.33
            48 -1.39  0.62
            72 -1.61  0.58
Shankhapark 24 -0.38  0.39
            48 -0.45  0.42
            72 -0.46  0.32
no-graph better in 12 of 12 cells


## 3. Classification ablation

In [6]:
# classification ablation, macro-F1 and class 4 recall
c = pd.DataFrame(json.load(open(P / "ablate_clf_summary.json")))
print(c.groupby(["horizon", "arm"])[["macro_f1", "c4_recall"]].mean().round(3))

q = c.pivot_table(index=["station", "horizon", "seed"], columns="arm", values="macro_f1")
q["diff"] = q["graph"] - q["nograph"]
print()
print(q.groupby(["station", "horizon"])["diff"].agg(["mean", "std"]).round(3))

                 macro_f1  c4_recall
horizon arm                         
+24h    graph       0.723      1.000
        nograph     0.712      0.942
+48h    graph       0.665      1.000
        nograph     0.659      1.000
+72h    graph       0.603      1.000
        nograph     0.588      1.000

                      mean    std
station     horizon              
Bhaktapur   +24h     0.068  0.041
            +48h     0.080  0.025
            +72h     0.080  0.030
Khumaltar   +24h    -0.014  0.005
            +48h    -0.013  0.034
            +72h    -0.018  0.031
Ratnapark   +24h    -0.000  0.025
            +48h    -0.009  0.021
            +72h    -0.017  0.030
Shankhapark +24h    -0.012  0.015
            +48h    -0.032  0.065
            +72h     0.015  0.026


## 4. Classification: LSTM vs GNN-LSTM

In [7]:
# lstm vs gnn classification, 3 seeds
print(pd.read_json(P / "clf_gnn_vs_lstm_summary.json"))

        station   h  f1_lstm  sd_lstm  c4_lstm  f1_gnn  sd_gnn  c4_gnn  n4  \
0     Bhaktapur  24    0.755    0.014        1   0.773   0.001       1  44   
1     Bhaktapur  48    0.681    0.030        1   0.704   0.025       1  44   
2     Bhaktapur  72    0.569    0.049        1   0.641   0.027       1  34   
3     Khumaltar  24    0.605    0.018        1   0.592   0.008       1  10   
4     Khumaltar  48    0.637    0.033        1   0.642   0.009       1  23   
5     Khumaltar  72    0.591    0.024        1   0.609   0.013       1  23   
6     Ratnapark  24    0.660    0.019        1   0.685   0.020       1  46   
7     Ratnapark  48    0.569    0.007        1   0.583   0.015       1  47   
8     Ratnapark  72    0.503    0.017        1   0.503   0.036       1  39   
9   Shankhapark  24    0.829    0.014        1   0.841   0.011       1  58   
10  Shankhapark  48    0.735    0.041        1   0.731   0.027       1  59   
11  Shankhapark  72    0.636    0.058        1   0.662   0.022  

## 5. Significance tests

Wilcoxon test and bootstrap 95% CI over the 12 station x horizon cells. Only 3 seeds, so take these as rough.

In [8]:
cell = d.groupby(["station", "h", "model"])[["mae", "rmse"]].mean().unstack("model")
q_cell = q.groupby(["station", "horizon"])[["graph", "nograph"]].mean()
gl = pd.read_json(P / "clf_gnn_vs_lstm_summary.json")

tests = {
    "reg MAE: gnn - lstm": cell["mae"]["graph"] - cell["mae"]["lstm"],
    "reg MAE: nograph - graph": cell["mae"]["nograph"] - cell["mae"]["graph"],
    "reg RMSE: gnn - lstm": cell["rmse"]["graph"] - cell["rmse"]["lstm"],
    "reg RMSE: nograph - graph": cell["rmse"]["nograph"] - cell["rmse"]["graph"],
    "clf F1: graph - nograph": q_cell["graph"] - q_cell["nograph"],
    "clf F1: gnn - lstm": gl["diff"],
}
print(pd.DataFrame({k: paired_test(v) for k, v in tests.items()}).T)

                              n   mean  ci_low  ci_high  wilcoxon_p  positive
reg MAE: gnn - lstm        12.0  0.567   0.232    0.935      0.0093      10.0
reg MAE: nograph - graph   12.0 -0.930  -1.155   -0.714      0.0005       0.0
reg RMSE: gnn - lstm       12.0  0.021  -0.588    0.616      0.9697       6.0
reg RMSE: nograph - graph  12.0 -1.145  -1.463   -0.856      0.0005       0.0
clf F1: graph - nograph    12.0  0.011  -0.010    0.034      1.0000       4.0
clf F1: gnn - lstm         12.0  0.016   0.006    0.029      0.0098      10.0


## 6. Persistence baseline

Regression: forecast = PM2.5 at time t. Classification: class of the 24 h average ending at t. No training, so no seeds.

In [9]:
import pickle
from sklearn.metrics import f1_score

sc = pickle.load(open(P / "feature_scaler.pkl", "rb"))
scaler, cols = sc["scaler"], sc["feature_cols"]
i0 = cols.index("PM2.5_filled")
unscale = lambda x: x * scaler.data_range_[i0] + scaler.data_min_[i0]
epa_class = lambda v: np.digitize(np.floor(v * 10) / 10, [35.5, 55.5, 125.5]) + 1

base_reg, base_clf = [], []
for s in stations:
    z = np.load(P / f"sequences_{s}.npz", allow_pickle=True)
    m = z["split"] == "test"
    pm = unscale(z["X"][m][:, :, i0])  # back to ug/m3
    last, avg24 = pm[:, -1], pm[:, -24:].mean(axis=1)
    for k, h in enumerate(horizons):
        y = z["y_pm25"][m][:, k]
        base_reg.append({"station": s, "h": h, "mae": np.abs(y - last).mean(),
                         "rmse": np.sqrt(((y - last) ** 2).mean())})
        yc, pc = z["y_class"][m][:, k], epa_class(avg24)
        c4 = yc == 4
        base_clf.append({"station": s, "h": h,
                         "macro_f1": f1_score(yc, pc, average="macro", labels=[1, 2, 3, 4], zero_division=0),
                         "c4_recall": (pc[c4] == 4).mean() if c4.any() else np.nan})

base_reg = pd.DataFrame(base_reg).set_index(["station", "h"])
base_clf = pd.DataFrame(base_clf).set_index(["station", "h"])

In [10]:
# regression: models (3-seed mean) vs persistence
for metric in ("mae", "rmse"):
    t = cell[metric][["lstm", "graph", "nograph"]].copy()
    t["persistence"] = base_reg[metric]
    print(f"test {metric.upper()} per cell")
    print(t.round(2))
    print("mean:", t.mean().round(2).to_dict())
    print("cells beating persistence:", {m: int((t[m] < t["persistence"]).sum()) for m in ("lstm", "graph", "nograph")})
    print()

print(pd.DataFrame({f"reg {metric.upper()}: {m} - persistence": paired_test(cell[metric][m] - base_reg[metric])
                    for metric in ("mae", "rmse") for m in ("lstm", "graph", "nograph")}).T)

test MAE per cell
model            lstm  graph  nograph  persistence
station     h                                     
Bhaktapur   24  14.37  16.30    14.80        13.34
            48  15.69  17.05    16.03        15.15
            72  16.45  17.39    16.45        17.35
Khumaltar   24  10.28  11.02    10.18        10.40
            48  11.41  12.12    11.23        11.56
            72  12.75  12.97    12.37        13.42
Ratnapark   24  15.40  15.00    13.91        15.10
            48  16.73  16.53    15.14        16.55
            72  17.94  18.39    16.78        19.18
Shankhapark 24  12.47  12.85    12.47        12.13
            48  13.44  13.80    13.34        13.79
            72  13.54  13.86    13.40        15.40
mean: {'lstm': 14.21, 'graph': 14.77, 'nograph': 13.84, 'persistence': 14.45}
cells beating persistence: {'lstm': 7, 'graph': 5, 'nograph': 9}

test RMSE per cell
model            lstm  graph  nograph  persistence
station     h                                     
Bha

In [11]:
# classification: models (3-seed mean) vs persistence class
ng = q_cell.reset_index().assign(h=lambda t: t.horizon.str.strip("+h")).set_index(["station", "h"])["nograph"]
g2 = gl.assign(h=gl.h.astype(str)).set_index(["station", "h"])
clf_cmp = pd.DataFrame({"persistence": base_clf.macro_f1, "lstm": g2.f1_lstm, "gnn": g2.f1_gnn, "nograph": ng})
print("test macro-F1 per cell")
print(clf_cmp.round(3))
print("mean:", clf_cmp.mean().round(3).to_dict())
print("persistence Class-4 recall:", base_clf.c4_recall.round(3).to_dict())
print()
print(pd.DataFrame({f"clf F1: {m} - persistence": paired_test(clf_cmp[m] - clf_cmp["persistence"])
                    for m in ("lstm", "gnn", "nograph")}).T)

test macro-F1 per cell
                persistence   lstm    gnn  nograph
station     h                                     
Bhaktapur   24        0.836  0.755  0.773    0.705
            48        0.734  0.681  0.704    0.625
            72        0.658  0.569  0.641    0.560
Khumaltar   24        0.655  0.605  0.592    0.606
            48        0.620  0.637  0.642    0.656
            72        0.554  0.591  0.609    0.626
Ratnapark   24        0.669  0.660  0.685    0.686
            48        0.638  0.569  0.583    0.592
            72        0.516  0.503  0.503    0.520
Shankhapark 24        0.859  0.829  0.841    0.853
            48        0.798  0.735  0.731    0.763
            72        0.722  0.636  0.662    0.647
mean: {'persistence': 0.688, 'lstm': 0.648, 'gnn': 0.664, 'nograph': 0.653}
persistence Class-4 recall: {('Bhaktapur', '24'): 0.977, ('Bhaktapur', '48'): 0.977, ('Bhaktapur', '72'): 1.0, ('Khumaltar', '24'): 0.0, ('Khumaltar', '48'): 0.13, ('Khumaltar', '72'): 0.